In [2]:
import pandas as pd
from itertools import chain
data = pd.read_csv('archive/ner_dataset.csv', encoding= 'unicode_escape')
data.head()
from sklearn.model_selection import train_test_split
from keras_preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
import numpy as np
import tensorflow
from tensorflow.keras import Sequential, Model, Input
from tensorflow.keras.layers import LSTM, Embedding, Dense, TimeDistributed, Dropout, Bidirectional
from tensorflow.keras.utils import plot_model

In [3]:
pd.read_json('PANDA-dataset\dataset\PANDA-annotated-100k-shard0.jsonl', lines=True)

,original,rewrite,selected_word,perturbed_category,source_data_type
0,"Gregg and his family owned 80% of the company,...",Mrs. Gregg and her family owned 80% of the com...,his,Woman,wiki
1,"Gregg and his family owned 80% of the company,...",Gregg and them family owned 80% of the company...,his,Non-Binary,wiki
2,"Gregg and his family owned 80% of the company,...","Gregg and her family owned 80% of the company,...",his,Woman,wiki
3,"Gregg and his family owned 80% of the company,...",Gregg and them family owned 80% of the company...,his,Non-Binary,wiki
4,"Gregg and his family owned 80% of the company,...","Gregg and his family owned 80% of the company,...",European,Hispanic or Latino,wiki
...,...,...,...,...,...
19995,"This is a nervy , risky film , and Villeneuve ...","This is a nervy, risky film, and Villeneuve ha...",Bibi,Non-Binary,sst
19996,"Unfortunately , that 's precisely what Arthur ...","Unfortunately, that's precisely what Alice Don...",Arthur,Woman,sst
19997,"Unfortunately , that 's precisely what Arthur ...","Unfortunately, that's precisely what Luna Dong...",Arthur,Non-Binary,sst
19998,The best part about `` Gangs '' was Daniel Day...,"The best part about ""Gangs"" was Daniel Day-Lewis.",Day-Lewis,Man,sst


In [4]:
def get_dict_map(data, token_or_tag):
    tok2idx = {}
    idx2tok = {}
    
    if token_or_tag == 'token':
        vocab = list(set(data['Word'].to_list()))
    else:
        vocab = list(set(data['Tag'].to_list()))
    
    idx2tok = {idx:tok for  idx, tok in enumerate(vocab)}
    tok2idx = {tok:idx for  idx, tok in enumerate(vocab)}
    return tok2idx, idx2tok

In [5]:
token2idx, idx2token = get_dict_map(data, 'token')
tag2idx, idx2tag = get_dict_map(data, 'tag')

In [6]:
token2idx

{'Cozumel': 0,
 'basing': 1,
 'reaffirmed': 2,
 'Horizons': 3,
 'Gotovina': 4,
 'blast': 5,
 'U2': 6,
 'compensated': 7,
 'knowledge-based': 8,
 'Hogg': 9,
 'Culture': 10,
 'Denmark-Sweden': 11,
 'forgo': 12,
 'Wikipedia': 13,
 '1,000-year': 14,
 'encouragement': 15,
 '68-year-old': 16,
 'unwanted': 17,
 'Hendarso': 18,
 'Garza': 19,
 'third-ranked': 20,
 'award': 21,
 '125': 22,
 'Flavia': 23,
 'Ipsos': 24,
 'Nalchik': 25,
 'Cambodian': 26,
 'Casablanca': 27,
 'kilometer-per-hour': 28,
 'paddy': 29,
 'bbl/day': 30,
 'uncalculated': 31,
 'medications': 32,
 'MONUC': 33,
 'Gwyneth': 34,
 'national-level': 35,
 'Shaktoi': 36,
 'detained': 37,
 'Donald': 38,
 'sanctuaries': 39,
 'react': 40,
 'ideological': 41,
 'newscaster': 42,
 'casual': 43,
 'worrisome': 44,
 'Aufdenblatten': 45,
 'Dai': 46,
 'Tombstone': 47,
 'ability': 48,
 'damaging': 49,
 'Qom': 50,
 'Stiff': 51,
 'Eduard': 52,
 'eight-tenths': 53,
 'Acting': 54,
 'laments': 55,
 'Spaniard': 56,
 'brokers': 57,
 'pro-Shiite': 58,


In [7]:
data['Word_idx'] = data['Word'].map(token2idx)
data['Tag_idx'] = data['Tag'].map(tag2idx)
data.head()

,Sentence #,Word,POS,Tag,Word_idx,Tag_idx
0,Sentence: 1,Thousands,NNS,O,9815,7
1,NaN,of,IN,O,3446,7
2,NaN,demonstrators,NNS,O,15977,7
3,NaN,have,VBP,O,2142,7
4,NaN,marched,VBN,O,17605,7


In [8]:
# Fill na
data_fillna = data.ffill(axis=0)# Groupby and collect columns
data_group = data_fillna.groupby(
['Sentence #'],as_index=False
).agg(lambda x: list(x))# Visualise data
data_group.head()

,Sentence #,Word,POS,Tag,Word_idx,Tag_idx
0,Sentence: 1,"[Thousands, of, demonstrators, have, marched, ...","[NNS, IN, NNS, VBP, VBN, IN, NNP, TO, VB, DT, ...","[O, O, O, O, O, O, B-geo, O, O, O, O, O, B-geo...","[9815, 3446, 15977, 2142, 17605, 25172, 8397, ...","[7, 7, 7, 7, 7, 7, 13, 7, 7, 7, 7, 7, 13, 7, 7..."
1,Sentence: 10,"[Iranian, officials, say, they, expect, to, ge...","[JJ, NNS, VBP, PRP, VBP, TO, VB, NN, TO, JJ, J...","[B-gpe, O, O, O, O, O, O, O, O, O, O, O, O, O,...","[568, 30583, 31575, 560, 24831, 547, 13975, 34...","[9, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, ..."
2,Sentence: 100,"[Helicopter, gunships, Saturday, pounded, mili...","[NN, NNS, NNP, VBD, JJ, NNS, IN, DT, NNP, JJ, ...","[O, O, B-tim, O, O, O, O, O, B-geo, O, O, O, O...","[2293, 32816, 26813, 3336, 13195, 5091, 20016,...","[7, 7, 3, 7, 7, 7, 7, 7, 13, 7, 7, 7, 7, 7, 1,..."
3,Sentence: 1000,"[They, left, after, a, tense, hour-long, stand...","[PRP, VBD, IN, DT, NN, JJ, NN, IN, NN, NNS, .]","[O, O, O, O, O, O, O, O, O, O, O]","[31231, 12531, 2934, 13633, 32837, 33693, 2000...","[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7]"
4,Sentence: 10000,"[U.N., relief, coordinator, Jan, Egeland, said...","[NNP, NN, NN, NNP, NNP, VBD, NNP, ,, NNP, ,, J...","[B-geo, O, O, B-per, I-per, O, B-tim, O, B-geo...","[24201, 15951, 18746, 29290, 18231, 28777, 115...","[13, 7, 7, 12, 16, 7, 3, 7, 13, 7, 9, 7, 9, 7,..."


In [9]:
def get_pad_train_test_val(data_group, data):

    #get max token and tag length
    n_token = len(list(set(data['Word'].to_list())))
    n_tag = len(list(set(data['Tag'].to_list())))

    #Pad tokens (X var)    
    tokens = data_group['Word_idx'].tolist()
    maxlen = max([len(s) for s in tokens])
    pad_tokens = pad_sequences(tokens, maxlen=maxlen, dtype='int32', padding='post', value= n_token - 1)

    #Pad Tags (y var) and convert it into one hot encoding
    tags = data_group['Tag_idx'].tolist()
    pad_tags = pad_sequences(tags, maxlen=maxlen, dtype='int32', padding='post', value= tag2idx["O"])
    n_tags = len(tag2idx)
    pad_tags = [to_categorical(i, num_classes=n_tags) for i in pad_tags]
    
    #Split train, test and validation set
    tokens_, test_tokens, tags_, test_tags = train_test_split(pad_tokens, pad_tags, test_size=0.1, train_size=0.9, random_state=2020)
    train_tokens, val_tokens, train_tags, val_tags = train_test_split(tokens_,tags_,test_size = 0.25,train_size =0.75, random_state=2020)

    print(
        'train_tokens length:', len(train_tokens),
        '\ntrain_tokens length:', len(train_tokens),
        '\ntest_tokens length:', len(test_tokens),
        '\ntest_tags:', len(test_tags),
        '\nval_tokens:', len(val_tokens),
        '\nval_tags:', len(val_tags),
    )
    
    return train_tokens, val_tokens, test_tokens, train_tags, val_tags, test_tags

train_tokens, val_tokens, test_tokens, train_tags, val_tags, test_tags = get_pad_train_test_val(data_group, data)

train_tokens length: 32372 
train_tokens length: 32372 
test_tokens length: 4796 
test_tags: 4796 
val_tokens: 10791 
val_tags: 10791


In [10]:
from numpy.random import seed
seed(1)
tensorflow.random.set_seed(2)

In [12]:
input_dim = len(list(set(data['Word'].to_list())))+1
output_dim = 64
input_length = max([len(s) for s in data_group['Word_idx'].tolist()])
n_tags = len(tag2idx)
print('input_dim: ', input_dim, '\noutput_dim: ', output_dim, '\ninput_length: ', input_length, '\nn_tags: ', n_tags)

input_dim:  35179 
output_dim:  64 
input_length:  104 
n_tags:  17


In [13]:
def get_bilstm_lstm_model():
    model = Sequential()

    # Add Embedding layer
    model.add(Embedding(input_dim=input_dim, output_dim=output_dim, input_length=input_length))

    # Add bidirectional LSTM
    model.add(Bidirectional(LSTM(units=output_dim, return_sequences=True, dropout=0.2, recurrent_dropout=0.2), merge_mode = 'concat'))

    # Add LSTM
    model.add(LSTM(units=output_dim, return_sequences=True, dropout=0.5, recurrent_dropout=0.5))

    # Add timeDistributed Layer
    model.add(TimeDistributed(Dense(n_tags, activation="relu")))

    #Optimiser 
    optimizer = tensorflow.keras.optimizers.Adam(lr=0.01, decay=1e-6)

    # Compile model
    model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    model.summary()
    
    return model

In [14]:
def train_model(X, y, model):
    loss = list()
    for i in range(25):
        # fit model for one epoch on this sequence
        hist = model.fit(X, y, batch_size=1000, verbose=1, epochs=1, validation_split=0.2)
        loss.append(hist.history['loss'][0])
    return loss

In [15]:
results = pd.DataFrame()
model_bilstm_lstm = get_bilstm_lstm_model()
plot_model(model_bilstm_lstm)
results['with_add_lstm'] = train_model(train_tokens, np.array(train_tags), model_bilstm_lstm)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 104, 64)           2251456   
                                                                 
 bidirectional (Bidirectiona  (None, 104, 128)         66048     
 l)                                                              
                                                                 
 lstm_1 (LSTM)               (None, 104, 64)           49408     
                                                                 
 time_distributed (TimeDistr  (None, 104, 17)          1105      
 ibuted)                                                         
                                                                 
Total params: 2,368,017
Trainable params: 2,368,017
Non-trainable params: 0
_________________________________________________________________
You must install pydot (`pip install pydot`) a